**Before you start: File → Save a copy in Drive, and work in the copy that opens. A notebook opened from a link does not keep your changes.**

# Homework 1: a warehouse robot, step by step

Advanced Robotics Project, Obuda University, Antal Bejczy Center for Intelligent Robotics.
Individual work. **Deadline: 2 October 2026 (week 4)**, on Moodle.

A robot in a warehouse gets an order: visit three shelves, pick something at each, then bring everything to the packing station. In Parts 1 to 3 you see how the robot drives, steers and plans: you run, try and answer. In Parts 4 and 5 you write the robot's decisions. No robotics background is needed.

| Part | What you do | About |
|---|---|---|
| 1. Drive | move two sliders, answer one question | 10 min |
| 2. Steer to a goal | switch one thing off, answer one question | 15 min |
| 3. Plan around shelves | change two numbers, answer one question | 15 min |
| 4. The mission rules | write six short rules | 1 hour |
| 5. One complication | choose one, write its rules | 1 hour |
| 6. Video and description | run two cells, write one page | 1 hour |

**How it works**
- Run the cells from top to bottom with Shift+Enter. If something looks strange: *Runtime → Restart session and run all*.
- Answers go between the quotes, like `answer_1 = "a"`. The same cell tells you `Correct` or `Not yet`.
- Code goes only between the lines `# ---- your code ... ----` and `# ------...`.
- The **Background** notes are optional. They explain the ideas behind the code for those who want them; you can do the homework without them.

**Hand in on Moodle:** this notebook (File → Download → Download .ipynb), and the video and the one-page description from Part 6.

In [ ]:
# @title Setup: run this cell once. It loads the robot, the warehouse, the planner and the simulator. You do not need to read it.
import heapq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Polygon, Rectangle, Circle
from matplotlib.collections import LineCollection
from IPython.display import HTML
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, fixed

DT = 0.05                   # seconds per simulation step
COMPLICATION = 0            # 0 = none; you choose 1, 2 or 3 in Part 5

# ---------------- the robot ----------------
def wrap(a):
    """Fold an angle (radians) into -pi ... pi."""
    return (a + np.pi) % (2 * np.pi) - np.pi

def step(state, v, w, dt=DT):
    """Differential drive: forward along the heading, turn by w * dt, never sideways."""
    x, y, th = state
    return np.array([x + v * np.cos(th) * dt, y + v * np.sin(th) * dt, wrap(th + w * dt)])

def go_to_goal(state, goal, use_wrap=True, k_rho=0.8, k_alpha=2.5, v_max=1.0):
    """Proportional controller: distance error -> forward speed v, heading error -> turning rate w."""
    x, y, th = state
    dx, dy = goal[0] - x, goal[1] - y
    rho = np.hypot(dx, dy)
    alpha = np.arctan2(dy, dx) - th
    if use_wrap:
        alpha = wrap(alpha)
    v, w = min(k_rho * rho, v_max), k_alpha * alpha
    if abs(alpha) > np.pi / 2:              # goal behind the robot: turn on the spot first
        v = 0.0
    return v, w

def robot_patch(state, size=0.35, color="#D85A30", alpha=1.0):
    x, y, th = state
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    pts = np.array([[size, 0], [-size * 0.6, size * 0.5], [-size * 0.6, -size * 0.5]]) @ R.T + [x, y]
    return Polygon(pts, closed=True, color=color, alpha=alpha)

def _ghosts(ax, poses, every, color="#D85A30"):
    """The robot every `every` steps, fading in, and its last pose in full colour."""
    idx = list(range(0, len(poses), every))
    for j, k in enumerate(idx):
        ax.add_patch(robot_patch(poses[k], size=0.5, color=color, alpha=0.12 + 0.4 * j / max(1, len(idx) - 1)))
    ax.add_patch(robot_patch(poses[-1], size=0.5, color=color))

def show_drive(v=1.0, w=0.5):
    s, poses = np.array([0.0, 0.0, 0.0]), []
    for _ in range(int(4.0 / DT) + 1):
        poses.append(s)
        s = step(s, v, w)
    poses = np.array(poses)
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.plot(poses[:, 0], poses[:, 1], color="#bbb", lw=1)
    _ghosts(ax, poses, every=10)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_title(f"4 seconds with v = {v:.1f} m/s, w = {w:.1f} rad/s\none triangle every 0.5 s, the darkest is the end")
    plt.show()

def show_goal(gx=3.0, gy=2.0, start_heading_deg=0.0, use_wrap=True):
    s = np.array([0.0, 0.0, np.deg2rad(start_heading_deg)])
    poses, reached, turned = [s], None, 0.0
    for k in range(int(8.0 / DT)):
        if np.hypot(gx - s[0], gy - s[1]) < 0.1:
            reached = k * DT
            break
        new = step(s, *go_to_goal(s, (gx, gy), use_wrap=use_wrap))
        turned += abs(wrap(new[2] - s[2]))
        s = new
        poses.append(s)
    poses = np.array(poses)
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.plot(poses[:, 0], poses[:, 1], color="#bbb", lw=1)
    _ghosts(ax, poses, every=8)
    ax.plot(gx, gy, "*", ms=16, color="#2E9E5B")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    result = f"reached the goal after {reached:.1f} s" if reached is not None else "did not reach the goal in 8 s"
    ax.set_title(f"wrap {'on' if use_wrap else 'off'}: {result}\nthe robot turned {turned / (2 * np.pi):.1f} full circles")
    plt.show()

# ---------------- the warehouse and the planner ----------------
CELL = 0.5                  # metres per grid cell
W, H = 36, 24               # 36 x 24 cells = 18 m x 12 m

def make_warehouse():
    grid = np.zeros((H, W), dtype=int)                 # 0 = free, 1 = wall or shelf
    grid[0, :] = grid[-1, :] = grid[:, 0] = grid[:, -1] = 1
    for c in (5, 11, 17, 23):                          # four shelf columns with a cross aisle
        grid[5:11, c:c + 2] = 1
        grid[15:20, c:c + 2] = 1
    return grid

def inflate(grid, radius_cells=1):
    """Grow every blocked cell by the robot's radius, so the planner can treat the robot as a point."""
    out = grid.copy()
    for r, c in np.argwhere(grid == 1):
        out[max(0, r - radius_cells):r + radius_cells + 1, max(0, c - radius_cells):c + radius_cells + 1] = 1
    return out

def to_cell(p):   return (int(round(p[1] / CELL)), int(round(p[0] / CELL)))
def to_world(rc): return np.array([rc[1] * CELL, rc[0] * CELL])
def inside(rc):   return 0 <= rc[0] < H and 0 <= rc[1] < W

def astar(grid, start_rc, goal_rc, explored=None):
    """A* on an 8-connected grid: a list of (row, col) cells, or None if there is no path."""
    if not (inside(start_rc) and inside(goal_rc)) or grid[start_rc] == 1 or grid[goal_rc] == 1:
        return None
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
    h = lambda a, b: np.hypot(a[0] - b[0], a[1] - b[1])
    open_set, came, g = [(h(start_rc, goal_rc), 0.0, start_rc)], {}, {start_rc: 0.0}
    while open_set:
        _, gc, cur = heapq.heappop(open_set)
        if explored is not None:
            explored.append(cur)       # for the drawing: every cell A* looked at
        if cur == goal_rc:
            path = [cur]
            while cur in came:
                cur = came[cur]
                path.append(cur)
            return path[::-1]
        for dr, dc in nbrs:
            n = (cur[0] + dr, cur[1] + dc)
            if not inside(n) or grid[n] == 1:
                continue
            ng = gc + np.hypot(dr, dc)
            if ng < g.get(n, np.inf):
                g[n], came[n] = ng, cur
                heapq.heappush(open_set, (ng + h(n, goal_rc), ng, n))
    return None

GRID = make_warehouse()
CSPACE = inflate(GRID)
START = (1.0, 1.0, 0.0)
LOCATIONS = {"A": (4.0, 4.0), "B": (7.0, 8.5), "C": (10.0, 3.0), "D": (13.5, 9.0), "E": (4.5, 9.5),
             "F": (10.5, 6.5), "PACKING": (16.0, 2.0), "CHARGER": (1.0, 6.5)}
SHELVES = ["A", "B", "C", "D", "E", "F"]
for _name, _p in list(LOCATIONS.items()) + [("START", START[:2])]:
    assert CSPACE[to_cell(_p)] == 0, f"{_name} is not free on the inflated map"
GOAL_TOL, WP_TOL = 0.15, 0.25   # metres: close enough to a goal / to a waypoint

def plan_path(state, goal, grid):
    cells = astar(grid, to_cell(state[:2]), to_cell(goal))
    if cells is None:
        return None
    return [to_world(c) for c in cells][1:] or [np.array(goal, float)]

def path_length(a, b, grid=CSPACE):
    cells = astar(grid, to_cell(a), to_cell(b))
    if cells is None:
        return float("inf")
    pts = np.array([to_world(c) for c in cells])
    return float(np.hypot(*np.diff(pts, axis=0).T).sum()) if len(pts) > 1 else 0.0

def distance(state, goal):
    return float(np.hypot(goal[0] - state[0], goal[1] - state[1]))

def drive_along(state, path, goal, dt=DT):
    """One time step along a planned path: drop passed waypoints, aim about 1 m ahead."""
    d = lambda p: np.hypot(*(np.asarray(p) - state[:2]))
    while len(path) > 1 and (d(path[0]) < WP_TOL or d(path[1]) < d(path[0])):
        path = path[1:]
    target = path[min(2, len(path) - 1)] if len(path) > 1 else goal
    v, w = go_to_goal(state, target)
    if abs(wrap(np.arctan2(target[1] - state[1], target[0] - state[0]) - state[2])) > np.pi / 4:
        v = 0.0                             # more than 45 degrees off: turn on the spot first
    return step(state, v, w, dt), path

EXTENT = [-CELL / 2, (W - 0.5) * CELL, -CELL / 2, (H - 0.5) * CELL]

def draw_map(ax):
    ax.imshow(GRID, cmap="Greys", origin="lower", alpha=0.8, extent=EXTENT)
    for n in SHELVES:
        ax.plot(*LOCATIONS[n], "o", ms=7, color="#2E9E5B")
        ax.annotate(n, LOCATIONS[n], xytext=(5, 4), textcoords="offset points", fontsize=11, weight="bold")
    ax.plot(*LOCATIONS["PACKING"], "s", ms=12, color="#378ADD")
    ax.annotate("packing", LOCATIONS["PACKING"], xytext=(-26, 10), textcoords="offset points")
    ax.plot(*LOCATIONS["CHARGER"], "D", ms=10, color="#E0A800")
    ax.annotate("charger", LOCATIONS["CHARGER"], xytext=(7, 5), textcoords="offset points")
    ax.set_xlim(EXTENT[0], EXTENT[1]); ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")

ROBOT_RADIUS = 0.3          # metres: the robot's body, for the drawing in Part 3

def show_plan(goal, inflate_cells=1):
    cspace = inflate(GRID, inflate_cells)
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    draw_map(ax)
    ax.imshow(np.ma.masked_where(cspace == GRID, cspace), cmap="Oranges", vmin=0, vmax=1.3,
              origin="lower", alpha=0.6, extent=EXTENT)      # the grown area, on top of the map
    ax.plot(*START[:2], "^", ms=10, color="#D85A30")
    ax.plot(*goal, "*", ms=16, color="#2E9E5B", zorder=5)
    explored = []
    cells = astar(cspace, to_cell(START[:2]), to_cell(goal), explored)
    if cells is None:
        ax.set_title("no path")
        print("A* found no path: the goal is inside a blocked cell (or outside the map), so astar returned None.")
        plt.show()
        return
    seen = np.zeros(GRID.shape)
    for rc in explored:
        seen[rc] = 1.0
    ax.imshow(np.ma.masked_where(seen == 0, seen), cmap="Blues", vmin=0, vmax=2.5, origin="lower", alpha=0.45, extent=EXTENT)
    pts = np.array([to_world(c) for c in cells])
    blocked = np.array([to_world(rc) for rc in np.argwhere(GRID == 1)])
    hits = 0
    for p in pts:
        gap = np.hypot(np.maximum(np.abs(blocked[:, 0] - p[0]) - CELL / 2, 0),
                       np.maximum(np.abs(blocked[:, 1] - p[1]) - CELL / 2, 0)).min()
        hit = gap < ROBOT_RADIUS
        hits += hit
        ax.add_patch(Circle(p, ROBOT_RADIUS, color="#D62728" if hit else "#378ADD", alpha=0.6 if hit else 0.13, lw=0))
    ax.plot(pts[:, 0], pts[:, 1], "-", lw=1.5, color="#378ADD")
    body = (f"the robot's body would hit a shelf or wall at {hits} points (red)" if hits
            else "the robot's body never touches a shelf or wall")
    ax.set_title(f"A* looked at {len(set(explored))} cells (light blue) and found a path of "
                 f"{path_length(START[:2], goal, cspace):.1f} m\n{body}")
    plt.show()

# ---------------- the job and the complications ----------------
ORDER = ["A", "B", "D"]                         # the shelves, in this sequence
ORDERS = {"order 1": ["D", "F"], "order 2": ["B", "E"], "order 3": ["A", "C"]}   # complication 3
PICK_TIME = 2.0                                 # seconds to pick, and to hand over at the packing station
T_MAX = 300.0                                   # the simulation stops here in any case
STATES = ["IDLE", "PLAN", "MOVE", "PICK", "DELIVER", "CHARGE", "DONE"]
BATTERY_START, BATTERY_LOW, BATTERY_FULL = 35.0, 25.0, 100.0    # percent (complication 1)
DRAIN_PER_METRE, CHARGE_RATE = 2.0, 20.0        # percent per metre driven, percent per second charging
PALLET_CELLS, PALLET_TIME = [(4, 8), (4, 9)], 2.0               # complication 2
_with_pallet = GRID.copy()
for _rc in PALLET_CELLS:
    _with_pallet[_rc] = 1
CSPACE_WITH_PALLET = inflate(_with_pallet)

def grid_at(t):
    return CSPACE_WITH_PALLET if COMPLICATION == 2 and t >= PALLET_TIME else CSPACE

def path_is_blocked(path, grid):
    return any(grid[to_cell(p)] == 1 for p in path)

class Robot:
    """The questions your rules can ask. Each one answers True or False."""
    def __init__(self, sim): self._sim = sim
    def has_order(self):       return self._sim.has_order()
    def path_found(self):      return len(self._sim.path) > 0
    def at_goal(self):         return self._sim.goal is not None and distance(self._sim.state, self._sim.goal) < GOAL_TOL
    def goal_is_shelf(self):   return self._sim.goal_name in SHELVES
    def goal_is_packing(self): return self._sim.goal_name == "PACKING"
    def goal_is_charger(self): return self._sim.goal_name == "CHARGER"
    def done_waiting(self):    return self._sim.timer >= PICK_TIME
    def battery_low(self):     return COMPLICATION == 1 and self._sim.battery < BATTERY_LOW
    def battery_full(self):    return self._sim.battery >= BATTERY_FULL
    def path_blocked(self):    return len(self._sim.path) > 0 and path_is_blocked(self._sim.path, self._sim.grid)

class _Sim:
    def __init__(self):
        self.state, self.mode, self.t = np.array(START, float), "IDLE", 0.0
        self.battery, self.grid, self.timer = BATTERY_START, CSPACE, 0.0
        self.goal, self.goal_name, self.path = None, None, []
        self.started, self.to_visit, self.order_name = False, [], None
        self.orders_left = {n: list(s) for n, s in ORDERS.items()} if COMPLICATION == 3 else {}
        self.picks, self.deliveries, self.charged, self.counted = [], [], False, False
        self.events = []                # (t, what, name, position): for the drawings

    def has_order(self):
        return len(self.orders_left) > 0 if COMPLICATION == 3 else not self.started

    def start_order(self):
        if COMPLICATION == 3:
            distances = {n: path_length(self.state[:2], LOCATIONS[s[0]], self.grid) for n, s in self.orders_left.items()}
            name = choose_next_order(dict(distances))
            if name not in self.orders_left:
                print(f"choose_next_order returned {name!r}, which is not an order that is left; starting {next(iter(self.orders_left))!r}.")
                name = next(iter(self.orders_left))
            self.order_name, self.to_visit = name, self.orders_left.pop(name)
        else:
            self.order_name, self.to_visit = "the order", list(ORDER)
        self.started = True

    def enter(self, new, old):
        if new == "PLAN":
            if old == "IDLE" and self.has_order():
                self.start_order()
            if COMPLICATION == 1 and self.battery < BATTERY_LOW:
                self.goal_name = "CHARGER"
            else:
                self.goal_name = self.to_visit[0] if self.to_visit else "PACKING"
            self.goal = LOCATIONS[self.goal_name]
            self.path = plan_path(self.state, self.goal, self.grid) or []
        if new in ("PICK", "DELIVER", "CHARGE"):
            self.timer, self.counted = 0.0, False

    def act(self):
        if self.mode == "PLAN" and not self.path:
            self.path = plan_path(self.state, self.goal, self.grid) or []
        elif self.mode == "MOVE" and self.path:
            self.state, self.path = drive_along(self.state, self.path, self.goal)
        elif self.mode in ("PICK", "DELIVER"):
            self.timer += DT
            if self.timer >= PICK_TIME and not self.counted:
                self.counted = True
                if self.mode == "PICK" and self.to_visit and self.goal_name == self.to_visit[0]:
                    self.picks.append(self.to_visit.pop(0))
                    self.events.append((self.t, "pick", self.picks[-1], self.state[:2].copy()))
                if self.mode == "DELIVER" and self.goal_name == "PACKING":
                    self.deliveries.append(self.order_name)
                    self.events.append((self.t, "deliver", self.order_name, self.state[:2].copy()))
        elif self.mode == "CHARGE" and distance(self.state, LOCATIONS["CHARGER"]) < 0.3:
            self.battery = min(BATTERY_FULL, self.battery + CHARGE_RATE * DT)
            self.charged = True

class MissionLog:
    def __init__(self): self.t, self.states, self.modes, self.battery, self.paths, self.sim = [], [], [], [], [], None
    def record(self, sim):
        self.t.append(sim.t); self.states.append(sim.state.copy()); self.modes.append(sim.mode)
        self.battery.append(sim.battery); self.paths.append(np.array(sim.path, float).reshape(-1, 2))

def next_mode(mode, robot):           # replaced by your rules in Part 4
    return mode

def complication_rules(mode, robot):  # replaced in Part 5
    return None

def choose_next_order(distances):     # replaced in Part 5
    return next(iter(distances))

def run_warehouse():
    """Run the robot with your rules until DONE or T_MAX. Returns the log."""
    sim, log = _Sim(), MissionLog()
    robot = Robot(sim)
    while sim.t < T_MAX:
        sim.grid = grid_at(sim.t)
        log.record(sim)
        old = sim.mode
        try:
            new = complication_rules(old, robot) if COMPLICATION else None
            if new is None:
                new = next_mode(old, robot)
        except Exception as err:
            print(f"Stopped at t = {sim.t:.1f} s in {old}: a rule raised {type(err).__name__}: {err}")
            break
        new = old if new is None else new
        if new not in STATES:
            print(f"Stopped at t = {sim.t:.1f} s: a rule returned {new!r}, but the states are {', '.join(STATES)}.")
            break
        if new != old:
            sim.enter(new, old)
            sim.mode = new
        if sim.mode == "DONE":
            break
        before = sim.state[:2].copy()
        sim.act()
        if COMPLICATION == 1:
            sim.battery = max(0.0, sim.battery - DRAIN_PER_METRE * float(np.hypot(*(sim.state[:2] - before))))
            if sim.battery <= 0.0:
                print(f"The battery ran empty at t = {sim.t:.1f} s: the robot stops.")
                break
        sim.t += DT
    log.record(sim)
    log.sim = sim
    return log

# ---------------- summary, plots, video ----------------
STATE_COLORS = {"IDLE": "#9E9E9E", "PLAN": "#D62728", "MOVE": "#378ADD", "PICK": "#2E9E5B",
                "DELIVER": "#8E44AD", "CHARGE": "#E0A800", "DONE": "#333333"}
TRIP_COLORS = ["#1f77b4", "#ff7f0e", "#9467bd", "#17becf", "#8c564b", "#e377c2", "#bcbd22"]

def _trip_colors(log):
    """One colour per trip: a new trip starts every time the robot enters PLAN."""
    trip, out = -1, []
    for k, m in enumerate(log.modes):
        if m == "PLAN" and (k == 0 or log.modes[k - 1] != "PLAN"):
            trip += 1
        out.append(TRIP_COLORS[max(trip, 0) % len(TRIP_COLORS)])
    return out

def _draw_pallet(ax, visible=True):
    (r0, c0), (r1, c1) = PALLET_CELLS[0], PALLET_CELLS[-1]
    return ax.add_patch(Rectangle((c0 * CELL - CELL / 2, r0 * CELL - CELL / 2), (c1 - c0 + 1) * CELL,
                                  (r1 - r0 + 1) * CELL, color="#8B5A2B", visible=visible, zorder=3))

def _last_stay(log):
    k = len(log.modes) - 1
    while k > 0 and log.modes[k - 1] == log.modes[-1]:
        k -= 1
    return log.modes[-1], log.t[-1] - log.t[k]

def summarize(log):
    """What the robot did. 'order complete: yes' also needs the chosen complication to work."""
    s, notes, handled = log.sim, [], True
    print(f"simulated time: {log.t[-1]:.1f} s, final state: {log.modes[-1]}")
    print("shelves picked, in this sequence:", ", ".join(s.picks) if s.picks else "none")
    print("deliveries at the packing station:", len(s.deliveries))
    if COMPLICATION == 1:
        lowest = min(log.battery)
        print(f"lowest battery: {lowest:.0f} %, charged on the way: {'yes' if s.charged else 'no'}")
        handled = s.charged and lowest > 0.0
        if not s.charged:
            notes.append("The robot never charged. Complication 1 needs its three rules in complication_rules.")
    if COMPLICATION == 2:
        (r0, c0), (r1, c1) = PALLET_CELLS[0], PALLET_CELLS[-1]
        x0, x1, y0, y1 = c0 * CELL - CELL / 2, c1 * CELL + CELL / 2, r0 * CELL - CELL / 2, r1 * CELL + CELL / 2
        gaps = [np.hypot(max(x0 - p[0], 0.0, p[0] - x1), max(y0 - p[1], 0.0, p[1] - y1))
                for t, p in zip(log.t, log.states) if t >= PALLET_TIME]
        gap = float(min(gaps)) if gaps else float("inf")
        print(f"closest distance to the pallet: {gap:.2f} m")
        handled = gap >= 0.25
        if not handled:
            notes.append("The robot drove into the pallet: it kept following a blocked path.")
    if COMPLICATION == 3:
        closest, left, here = [], dict(ORDERS), START[:2]
        while left:
            name = min(left, key=lambda n: path_length(here, LOCATIONS[left[n][0]]))
            closest.append(name)
            del left[name]
            here = LOCATIONS["PACKING"]
        print("orders delivered, in this sequence:", ", ".join(s.deliveries) if s.deliveries else "none")
        handled = s.deliveries == closest
        complete = log.modes[-1] == "DONE" and handled
        if len(s.deliveries) < len(ORDERS):
            notes.append("Not every order was delivered.")
        elif not handled:
            notes.append(f"The closest order has to come first each time: {', '.join(closest)}.")
    else:
        complete = log.modes[-1] == "DONE" and s.picks == ORDER and len(s.deliveries) == 1 and handled
        if log.modes[-1] == "DONE" and (s.picks != ORDER or len(s.deliveries) != 1):
            notes.append(f"The robot reached DONE, but it should first pick {', '.join(ORDER)} and then deliver once.")
    print("order complete:", "yes" if complete else "not yet")
    if not complete:
        mode, stay = _last_stay(log)
        recent = [m for t, m in zip(log.t, log.modes) if t >= log.t[-1] - 20.0]
        switches = sum(a != b for a, b in zip(recent, recent[1:]))
        timed_out = log.t[-1] >= T_MAX - DT / 2
        if COMPLICATION == 1 and min(log.battery) <= 0.0:
            notes.insert(0, "The battery ran empty while driving. The robot has to head for the charger as soon as "
                            "the battery is low, also in the middle of a trip.")
        elif mode != "DONE" and timed_out and switches >= 3:
            notes.insert(0, f"In the last 20 s the robot kept switching between {', '.join(dict.fromkeys(recent))}: "
                            "one of your rules sends it to the wrong state.")
        elif mode != "DONE":
            notes.insert(0, f"The robot spent the last {stay:.0f} s in {mode}: look at the rules that leave {mode}.")
        for n in notes:
            print("  " + n)
    return complete

def plot_states(log):
    """The run at a glance: the path seen from above, and the state over time."""
    xy = np.array([p[:2] for p in log.states])
    rows = 2 if COMPLICATION == 1 else 1
    fig = plt.figure(figsize=(14, 5.4))
    gs = fig.add_gridspec(rows, 2, width_ratios=[1.3, 1], hspace=0.45, wspace=0.15)
    ax = fig.add_subplot(gs[:, 0])
    draw_map(ax)
    if COMPLICATION == 2:
        _draw_pallet(ax)
    colors = _trip_colors(log)
    ax.add_collection(LineCollection(np.stack([xy[:-1], xy[1:]], axis=1), colors=colors[1:], linewidths=3, zorder=4))
    for k in range(1, len(log.modes)):
        if log.modes[k] == "PLAN" and log.modes[k - 1] == "MOVE":
            ax.plot(*xy[k], "X", ms=13, color="#D62728", mec="white", zorder=6)
        if log.modes[k] == "CHARGE" and log.modes[k - 1] != "CHARGE":
            ax.plot(*xy[k], "D", ms=17, mfc="none", mec="#E0A800", mew=3, zorder=6)
    for n, (t, what, name, p) in enumerate([ev for ev in log.sim.events if ev[1] == "pick"], start=1):
        ax.annotate(str(n), p, xytext=(-13, -17), textcoords="offset points", fontsize=9, color="white", weight="bold",
                    ha="center", va="center", bbox=dict(boxstyle="circle,pad=0.25", fc="#2E9E5B", ec="white"), zorder=7)
    for t, what, name, p in [ev for ev in log.sim.events if ev[1] == "deliver"]:
        ax.plot(*p, "s", ms=20, mfc="none", mec="#8E44AD", mew=3, zorder=6)
    ax.plot(*xy[-1], "o", ms=8, color="#333", zorder=8)
    ax.set_title("from above: one colour per trip, green numbers = picks in order,\nred X = replanned while driving")
    tl = fig.add_subplot(gs[0, 1])
    k0 = 0
    for k in range(1, len(log.modes) + 1):
        if k == len(log.modes) or log.modes[k] != log.modes[k0]:
            m = log.modes[k0]
            end = log.t[k] if k < len(log.t) else log.t[-1]
            tl.broken_barh([(log.t[k0], max(end - log.t[k0], 0.4))], (STATES.index(m) - 0.35, 0.7), facecolors=STATE_COLORS[m])
            k0 = k
    tl.step(log.t, [STATES.index(m) for m in log.modes], where="post", color="#bbb", lw=0.8, zorder=0)
    tl.set_yticks(range(len(STATES))); tl.set_yticklabels(STATES); tl.grid(alpha=0.3, axis="x")
    tl.set_title("the state over time")
    tl.set_xlabel("time [s]")
    if COMPLICATION == 1:
        b = fig.add_subplot(gs[1, 1], sharex=tl)
        b.plot(log.t, log.battery, color="#2E9E5B", lw=2)
        b.axhline(BATTERY_LOW, ls="--", color="#999")
        b.set_ylabel("battery [%]"); b.set_ylim(0, 105); b.grid(alpha=0.3); b.set_xlabel("time [s]")
    plt.show()

def animate_run(log, max_frames=200, interval=50, dpi=100):
    fig, ax = plt.subplots(figsize=(8.5, 5.8), dpi=dpi)
    draw_map(ax)
    pallet = _draw_pallet(ax, visible=False)
    rings = {n: ax.plot(*LOCATIONS[n], "o", ms=18, mfc="none", mec="#2E9E5B", mew=3, visible=False)[0] for n in SHELVES}
    every = max(1, len(log.t) // max_frames)
    frames = list(range(0, len(log.t), every)) + [len(log.t) - 1]
    xy = np.array([p[:2] for p in log.states])
    colors = _trip_colors(log)
    trail = LineCollection([], linewidths=3, zorder=4)
    ax.add_collection(trail)
    plan, = ax.plot([], [], "--", color="#555", lw=1.2, zorder=4)
    body = ax.add_patch(robot_patch(log.states[0], size=0.5))
    body.set_zorder(6)
    label = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top", fontsize=11, zorder=9,
                    bbox=dict(boxstyle="round", fc="white", alpha=0.9))
    picks = [(t, name) for t, what, name, p in log.sim.events if what == "pick"]

    def update(i):
        k, shown = frames[i], frames[:i + 1]
        pts = xy[shown]
        trail.set_segments(np.stack([pts[:-1], pts[1:]], axis=1) if len(pts) > 1 else [])
        trail.set_color([colors[j] for j in shown[1:]])
        plan.set_data(log.paths[k][:, 0], log.paths[k][:, 1])
        body.set_xy(robot_patch(log.states[k], size=0.5).get_xy())
        body.set_color(STATE_COLORS[log.modes[k]])
        pallet.set_visible(COMPLICATION == 2 and log.t[k] >= PALLET_TIME)
        done = [name for t, name in picks if t <= log.t[k]]
        for n, ring in rings.items():
            ring.set_visible(n in done)
        text = f"t = {log.t[k]:5.1f} s   state: {log.modes[k]}"
        if COMPLICATION == 1:
            text += f"   battery: {log.battery[k]:3.0f} %"
        label.set_text(text + f"\npicked: {', '.join(done) if done else '-'}")
        label.get_bbox_patch().set_edgecolor(STATE_COLORS[log.modes[k]])
        label.get_bbox_patch().set_linewidth(2.5)
        return (trail, plan, body, pallet, label, *rings.values())

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return anim

def watch(log):
    """A quick, smaller animation of a run, to watch in the notebook."""
    return HTML(animate_run(log, max_frames=100, dpi=75).to_jshtml())

def export_video(anim, basename="h1_warehouse", fps=20):
    """MP4 when ffmpeg is available (it is in Colab), a GIF otherwise. Returns the file name."""
    if animation.writers.is_available("ffmpeg"):
        try:
            anim.save(basename + ".mp4", writer=animation.FFMpegWriter(fps=fps))
            return basename + ".mp4"
        except Exception as err:
            print(f"MP4 export did not work ({type(err).__name__}), saving a GIF instead.")
    anim.save(basename + ".gif", writer="pillow", fps=fps)
    return basename + ".gif"

# ---------------- checks ----------------
import base64, json, zlib

def check_answer(number, answer):
    """Questions 1-3: prints Correct or Not yet, with a reason."""
    q = json.loads(zlib.decompress(base64.b64decode(_QUESTIONS)))[str(number)]
    a = str(answer).strip().strip("()").strip().lower()
    if a == "":
        print('Not yet: write your answer between the quotes, for example "a", and run this cell again.')
    elif a not in ("a", "b", "c"):
        print(f'Not yet: the answer is one letter, "a", "b" or "c", not {answer!r}.')
    elif a == q["correct"]:
        print("Correct: " + q["why"])
    else:
        print("Not yet: " + q["hints"].get(a, "try the experiment above once more."))

class _FakeRobot:
    def __init__(self, **facts): self._facts = facts
    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        return lambda: self._facts.get(name, False)

_RULE_TESTS = [
    (1, "IDLE -> PLAN", [("IDLE", {"has_order": True}, "PLAN"), ("IDLE", {}, "IDLE")]),
    (2, "PLAN -> MOVE", [("PLAN", {"path_found": True}, "MOVE"), ("PLAN", {}, "PLAN")]),
    (3, "MOVE -> PICK", [("MOVE", {"at_goal": True, "goal_is_shelf": True}, "PICK"), ("MOVE", {"goal_is_shelf": True}, "MOVE")]),
    (4, "MOVE -> DELIVER", [("MOVE", {"at_goal": True, "goal_is_packing": True}, "DELIVER"), ("MOVE", {"goal_is_packing": True}, "MOVE")]),
    (5, "PICK -> PLAN", [("PICK", {"done_waiting": True}, "PLAN"), ("PICK", {}, "PICK")]),
    (6, "DELIVER -> DONE", [("DELIVER", {"done_waiting": True}, "DONE"), ("DELIVER", {}, "DELIVER")]),
]

def check_mission_rules(rules):
    lines, working = [], 0
    for num, title, cases in _RULE_TESTS:
        problem = None
        for mode, facts, want in cases:
            when = " and ".join(f"robot.{k}() is True" for k in facts) or "every question answers False"
            try:
                got = rules(mode, _FakeRobot(**facts))
            except Exception as err:
                problem = f"in {mode}, your rules raised {type(err).__name__}: {err}"
                break
            got = mode if got is None else got
            if got != want:
                problem = f"in {mode}, when {when}, the next state should be {want}, but your rules give {got!r}"
                if want == mode:
                    problem += (". If a rule fires although its question is False, check that every question "
                                "ends with (): robot.at_goal(), not robot.at_goal")
                break
        working += problem is None
        lines.append(f"  rule {num} ({title}): " + ("works" if problem is None else problem))
    if working == len(_RULE_TESTS):
        print("Correct: all six rules work. Run the next cell to watch the whole order.")
    else:
        print(f"Not yet: {working} of {len(_RULE_TESTS)} rules work.")
    print("\n".join(lines))

# data for check_answer
_QUESTIONS = "eJyFVNtq20AQ/ZWDX3rBNk7zEgylBFpKoZ9QKKvVWNpmtSN2V1bUkH/vzKqRlZLQFyNZq3ObOXrYXG2OeNhYjpFslutNtdliM7aTXo8utwiME8fRxBqpJ6q3yC0hcsUZ1gRw8BPyEAPGliLBZaRsQp32+DafCHSmiORdTUjyM5op7ZWmdSGnIsAonWe+g8kFP0dnQuMpwTTGhSOKljM+4qDPJxjvlWcS/vKCXMcM4Z0F9SzYcAG1O51EltzUTj06DjO5Vco8S07q0nLX6fvidpG5LYiKL+bIJ/UTxHgfuR4sCRftN4+PAvfh3yDtKkgFaNh4eKeOMq5uDj+Gw6E6zGkm09FFH0zCbjmxxxdjW6Rudkw9hh4iseYx4ORdn7DAXyAqyiNRuBAVI7sVb+KSKQ8ZYzR9AWnJ1C40oBiF4dfQCXglWeM0CHkZ8lMe8wKk3gU5YuzdHreaXMM5C21BlFgNrDcpOYtqaHQckURmMGfXmKLTck0v7cIQsrN3GBL9VCyRSxnNvcx/d100NFPZBbo3NvtpnpOAz+Kyy16Gcyo3vedcKCoFfv3MUeJbe5OTkvuSrYYlBm5EiWVZ70/z3K//VyAN2IWTF7+h2f5lky2SSmShyGlFaTSxeXXVj55LiK5p5d6zDKYsS0v+TKvVjFTDumi1LanlcSmiTIbrCSMPvkarxeSO1EWzx9fIo3uOqKNetLwRLPeb4Mnoo8jclWboAUV9aWjSoN4Is2qQKk2w5KUzt++h1RaVplSypD8y4iDu3j79oeN4d8S6kZVrlv5a2uP76gOxNi07mGWsz2qN3sgHI2Uni0syLn3v8+sQl3il9lrpxz/xoLqO"

---
## Part 1: Drive

**Idea.** The robot has two wheels, one on each side. You control it with two numbers: the forward speed `v` in metres per second, and the turning rate `w`, how fast it turns (in radians per second; 1 radian is about 57°). There is no command for moving sideways.

**Try.** Run the cell and move the sliders:
1. `w = 0`. What path does the robot drive?
2. `v = 1` and `w = 1`. And now?
3. `v = 0` and `w = 1`. And now?

In [ ]:
interact(show_drive, v=FloatSlider(1.0, min=-1.5, max=1.5, step=0.1), w=FloatSlider(0.5, min=-2.0, max=2.0, step=0.1));

**Question 1.** With `v = 0` and `w = 1`, what does the robot do?
a) it drives a circle, b) it turns on the spot, c) it slides sideways

In [ ]:
answer_1 = ""      # "a", "b" or "c"
check_answer(1, answer_1)

> **Background (optional).** In one short time step `dt` the robot moves `v·dt` in the direction θ it faces, so `x` grows by `v·cos(θ)·dt` and `y` by `v·sin(θ)·dt`, and the heading grows by `w·dt`. Constant `v` and `w` drive a circle of radius `v / w`. A robot that cannot move sideways is called *nonholonomic*; that is why parallel parking takes several moves.

---
## Part 2: Steer to a goal

**Idea.** To reach a goal, the robot measures two errors: how far away the goal is, and how far it would have to turn to face it, the *heading error*. It drives faster when the goal is far and turns harder when the heading error is large. This is **feedback**: measure the error, correct, repeat.

Angles have a catch: 190° and −170° point the same way. The function `wrap` folds the heading error into −180°…180°, so that the robot always turns the short way.

**Try.**
1. Move the goal behind the robot, for example `gx = -3`, `gy = 1`. The robot turns on the spot first, then drives.
2. Untick `use_wrap`, set `gx = -3` and `gy = 0`, and read the title.

In [ ]:
interact(show_goal, gx=FloatSlider(3.0, min=-3.5, max=3.5, step=0.25), gy=FloatSlider(2.0, min=-3.5, max=3.5, step=0.25),
         start_heading_deg=FloatSlider(0, min=-180, max=180, step=15), use_wrap=Checkbox(True));

**Question 2.** With `use_wrap` off and the goal at `gx = -3`, `gy = 0`, what happens?
a) the robot drives straight to the goal, b) it turns the long way round but arrives, c) it keeps spinning and never arrives

In [ ]:
answer_2 = ""      # "a", "b" or "c"
check_answer(2, answer_2)

> **Background (optional).** The distance error is ρ = √(dx² + dy²), the heading error is α = atan2(dy, dx) − θ, and the two rules are `v = k_ρ·ρ` and `w = k_α·α`: *proportional* feedback, because the correction is proportional to the error. `atan2` returns the direction of (dx, dy) between −180° and 180°, and that edge is where the jump comes from.

---
## Part 3: Plan around shelves

**Idea.** The planner sees the floor as a grid of 0.5 m cells, each free or blocked (grey). It treats the robot as a single point, so first it grows every shelf by the robot's size. This is called *inflating* (orange). Then the **A\*** planner searches the free cells for the shortest path. The result is a list of waypoints for the robot to follow. The picture also shows the cells A\* looked at (light blue) and the robot's body along the path.

**Try.** Change the numbers in the cell and run it again:
1. `GOAL = (15.0, 10.0)`, or any other point you like.
2. `GOAL = (5.5, 4.0)`. This point is inside a shelf: what does the planner say?
3. `GOAL = LOCATIONS["D"]` and `INFLATE_CELLS = 0`. Look for red circles, and compare with `INFLATE_CELLS = 1`.

In [ ]:
GOAL = LOCATIONS["D"]      # the goal (x, y) in metres; LOCATIONS["D"] is shelf D
INFLATE_CELLS = 1          # how many cells every shelf grows by
show_plan(GOAL, INFLATE_CELLS)

**Question 3.** With `INFLATE_CELLS = 0` the path is shorter. What is wrong with it?
a) A\* needs much longer to find it, b) the robot's body would hit shelves and walls, c) the robot cannot reach shelf D

In [ ]:
answer_3 = ""      # "a", "b" or "c"
check_answer(3, answer_3)

> **Background (optional).** A\* keeps a list of cells to look at and always takes the one with the lowest *cost so far + straight-line guess of the distance left*. The guess never overestimates, so the first path that reaches the goal is the shortest. Other planners you may meet: Dijkstra (A\* without the guess), D\* Lite (fast replanning), RRT (random sampling, for arms and large spaces).

---
## Part 4: The mission rules

**Idea.** The robot's behaviour is a **state machine**: the robot is always in exactly one state, and **rules** decide when it switches to another. The simulator already does what each state means (plan, drive, pick). You write when to switch.

| State | What the robot does (given) | Leave when (your rule) |
|---|---|---|
| IDLE | waits | 1. an order is waiting → PLAN |
| PLAN | chooses the next goal (the next shelf, or the packing station once all shelves are picked) and plans a path | 2. a path was found → MOVE |
| MOVE | follows the path | 3. at the goal, and the goal is a shelf → PICK<br>4. at the goal, and the goal is the packing station → DELIVER |
| PICK | picks for 2 seconds | 5. picking is finished (`robot.done_waiting()`) → PLAN |
| DELIVER | hands the goods over for 2 seconds | 6. handing over is finished (`robot.done_waiting()`) → DONE |
| DONE | stops | |

**The questions your rules can ask.** Each answers `True` or `False`: `robot.has_order()`, `robot.path_found()`, `robot.at_goal()`, `robot.goal_is_shelf()`, `robot.goal_is_packing()`, `robot.done_waiting()`.

**What a rule looks like.** Two lines: a question, then the state to switch to. `and` joins two questions. An example from Part 5:
```python
if robot.at_goal() and robot.goal_is_charger():
    return "CHARGE"
```
Write the `if` where `pass` is, and indent `return` by four more spaces, as in the example. When no rule fires, the robot stays in its state.

In [ ]:
def next_mode(mode, robot):
    """The rules of the state machine: returns the state the robot switches to."""
    if mode == "IDLE":
        # rule 1: an order is waiting -> PLAN
        # ---- your code: 2 lines ----
        pass
        # ----------------------------
    elif mode == "PLAN":
        # rule 2: a path was found -> MOVE
        # ---- your code: 2 lines ----
        pass
        # ----------------------------
    elif mode == "MOVE":
        # rule 3: at the goal, and the goal is a shelf -> PICK
        # rule 4: at the goal, and the goal is the packing station -> DELIVER
        # ---- your code: 4 lines ----
        pass
        # ----------------------------
    elif mode == "PICK":
        # rule 5: picking is finished -> PLAN
        # ---- your code: 2 lines ----
        pass
        # ----------------------------
    elif mode == "DELIVER":
        # rule 6: handing over is finished -> DONE
        # ---- your code: 2 lines ----
        pass
        # ----------------------------
    return mode    # no rule fired: stay in the same state

check_mission_rules(next_mode)

**Run the whole order.** When all six rules work, the robot picks A, B and D and delivers them. The left picture shows the run from above, the right one the state over time: every switch is one of your rules firing.

In [ ]:
COMPLICATION = 0
log = run_warehouse()
summarize(log)
plot_states(log)

**Watch it drive.** Preparing the animation takes about half a minute. The video you hand in comes in Part 6.

In [ ]:
watch(log)

> **Background (optional).** State machines have organised the logic of robot cells for decades. Their weak point shows in Part 5: every new situation adds arrows, often out of many states. Larger systems therefore use *behaviour trees*, which organise the same decisions as a tree of small tasks; Nav2, the ROS 2 navigation software, is built that way.

---
## Part 5: One complication

Choose **one** complication, set `COMPLICATION` in the cell below, and write its rules in `complication_rules`. These rules are checked before the rules of Part 4, so each one also says in which state it applies, for example `if mode == "PICK" and robot.done_waiting():`.

**1. Battery.** The robot starts with 35 % battery, and driving uses 2 % per metre: not enough for the whole order. When the robot goes to PLAN with a low battery, the simulator makes the charger its goal. Questions: `robot.battery_low()`, `robot.battery_full()`, `robot.goal_is_charger()`, `robot.at_goal()`. Three rules: while moving, the battery is low and the goal is not the charger (`not robot.goal_is_charger()`) → PLAN; while moving, at the goal and the goal is the charger → CHARGE; while charging, the battery is full → PLAN.

**2. Blocked aisle.** Two seconds after the start, a pallet is dropped in an aisle the robot wants to use. When the robot goes to PLAN, the simulator plans a new path on the current map. Question: `robot.path_blocked()`. One rule: while moving, the path is blocked → PLAN.

**3. Several orders.** There are three orders. Your rule sends the robot back to IDLE after each delivery, and `choose_next_order` picks the order whose first shelf is closest along the path. Questions: `robot.done_waiting()`, `robot.has_order()`. One rule: while delivering, handing over is finished and an order is left → IDLE. Then replace the placeholder line in `choose_next_order` with `return min(distances, key=distances.get)`. `distances` is a dictionary like `{"order 1": 14.5, "order 2": 9.0, "order 3": 6.5}` (metres), and this line returns the name with the smallest distance.

The summary says `order complete: yes` only when your complication works: the robot charged and never ran empty (1), stayed at least 0.25 m away from the pallet (2), or delivered the closest order first each time (3).

In [ ]:
COMPLICATION = 1      # choose: 1 = battery, 2 = blocked aisle, 3 = several orders

def complication_rules(mode, robot):
    """The extra rules of your complication. They are checked before the rules of Part 4."""
    # ---- your code: 2 to 6 lines ----
    pass
    # ---------------------------------
    return None       # no extra rule fired: the rules of Part 4 decide

def choose_next_order(distances):
    """Complication 3: the name of the order to start next. distances = {order name: metres along the path}."""
    # ---- your code: 1 line ----
    return next(iter(distances))      # placeholder: simply the first order
    # ---------------------------

log = run_warehouse()
summarize(log)
plot_states(log)

---
## Part 6: The video and the description

The first cell animates the last run, so run the Part 5 cell just before it. The second cell saves it as a video: `h1_warehouse.mp4` in Colab, or a GIF. Each takes about a minute, so let it finish. Then download the file from the Files panel (the folder icon on the left).

In [ ]:
anim = animate_run(log)
HTML(anim.to_jshtml())

In [ ]:
video = export_video(anim)
print("saved", video, "- download it from the Files panel on the left")

**The one-page description** (PDF) answers four questions:
1. Which complication did you choose? Draw your state machine: the states as boxes, the rules as arrows (a photo of a hand drawing is fine). If you chose 2, also explain why going back to PLAN is enough.
2. Pick one moment of your video. Which state is the robot in, and which rule will make it leave that state?
3. What did not work at first, and how did you find out why?
4. Your robot handles one complication. What would get harder if all three happened at the same time?

**Before you hand in**
- *Runtime → Restart session and run all* finishes without errors.
- Questions 1 to 3 and the rule check in Part 4 say `Correct`.
- The summary in Part 5 says `order complete: yes`.
- The video is downloaded and the description is ready.